## 09_03a - Editing interattivo
Solo editing: generazione, RAGAS e report restano nella 09_03.
Prima di cominciare (dopo `pre_edit`): copia il sidecar con la cella di backup. Poi: cella 1a (cambia `QID`, modifica nel widget) e 1b per controllare. Le modifiche si salvano da sole nel sidecar.

In [1]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys, json, logging
from pathlib import Path
import shutil

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark
"""
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                    datefmt="%H:%M:%S", force=True)
"""
QDIR = project_root / "data/queries/colosseum"
sidecar_path = project_root / "data/processed/colosseum/colosseum_benchmark.json"
queries_path = QDIR / "eval_queries.json"          # file unico (42 query): serve a update_query(new_id=...)
pred_path, norm_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json"

handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)

bench = RagasBenchmark(
    handler, predetermined_path=pred_path, normal_path=norm_path,
    out_dir=project_root / "data/benchmark/colosseum",
    top_j=8, top_k=15, top_n=7,
)
print("!>> Handler e benchmark pronti")

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(


!>> Handler e benchmark pronti


In [ ]:
### BACKUP sidecar (una volta, prima di modificare)
shutil.copy(sidecar_path, sidecar_path.with_name("dst_benchmark_pre_edit.json"))

In [2]:
### 1a ### EDITING INTERATTIVO - scegli la query: retrieval + reranking + widget (modifiche salvate nel sidecar)
import ipywidgets as widgets
from IPython.display import display

QID = "AMB_01"# cambia id per passare alla query successiva
q = bench.prepare(QID, order_by=["DIST","TAGS"], manual_tags=True) # mostra anche i tag assegnati nei json 
print(f"[{QID}] {q['query']}\n\nREFERENCE:\n{q['reference']}")

PHASE_MAP = {"1 Retrieval (Vettoriale)": "vectorial", "2 Retrieval (Distanze)": "distance",
             "Top-J Reranked": "reranked", "All Reranked": "all_reranked"}
phase_selector = widgets.ToggleButtons(options=list(PHASE_MAP), description="Fase Grafo:", button_style="info")
output_area = widgets.Output()

def on_phase_change(change):
    with output_area:
        output_area.clear_output(wait=True)
        handler.update_widget_view(PHASE_MAP[change["new"]])
        display(handler.widget)

phase_selector.observe(on_phase_change, names="value")
handler.update_widget_view("vectorial")
display(phase_selector, output_area)
with output_area:
    display(handler.widget)

!>> Tag assegnati alla query:
   >>> Tag di Conversazione: ['Conservation']
   >>> Tag Manuali della Query: []
   >>> Assegnazione Tag Automatici disabilitata
<<< Lista completa: ['Conservation']
[AMB_01] What is the arena floor made of?

REFERENCE:
The partially rebuilt arena floor uses laminated wood, reinforced with carbon-fibre and aramid-fibre where needed, assembled with stainless steel plates and bolts so that it can be dismounted (reversibility). The ancient floor was wood covered with sand.


ToggleButtons(button_style='info', description='Fase Grafo:', options=('1 Retrieval (Vettoriale)', '2 Retrieva…

Output()

In [ ]:
# Print nodi e testo
query_text = getattr(handler, "query_text", q["query"] if "q" in locals() else "N/D")

if hasattr(handler, "widget") and handler.widget.global_tags:
    global_tags = list(handler.widget.global_tags.keys())
elif hasattr(handler, "sidecar_manager"):
    global_tags = list(handler.sidecar_manager.get_global_tags().keys())
else:
    global_tags = []

print("=" * 60)
print(f"[{QID}] {q['query']}\n\nREFERENCE:\n{q['reference']}\n")
print(f"Query TAGS: {handler.query_tags}\n")
print(f"GLOBAL TAGS: {global_tags}")
print("=" * 60 + "\n")

category_map = {}
records_sources = [
    getattr(handler, "all_reranked", []),
    getattr(handler, "top_j_reranked", []),
    getattr(handler, "distance_retrieved_records", [])
]

for record_list in records_sources:
    for rec in (record_list or []):
        if isinstance(rec, dict):
            cid = rec.get("chunk_id")
            cat = rec.get("category")
            if cid and cat is not None:
                category_map[cid] = cat

# Recupero nodi dal grafo mostrato nel widget
graph_nodes = handler.widget.graph_data.get("nodes", [])

print(f"X>> TAG E TESTI DEI CHUNK NEL GRAFO ({len(graph_nodes)} nodi):\n")

for node in graph_nodes:
    node_id = node.get("id")
    text = node.get("text", "Nessun testo associato.")
    # Considera sia i tag modificati dall'utente che quelli di default
    tags = node.get("user_tags") or node.get("tags") or []

    category = category_map.get(node_id)
    
    print(f"?>> Chunk ID: {node_id}")
    print(f"  T>>> Tag associati: {tags if tags else 'Nessuno'}")
    if category is not None:
        print(f"  C>>> Categoria: Categoria {category}")
    print(f"<< Testo:\n{text}")
    print("-" * 60)

In [ ]:
### 1b ### Dopo le modifiche: rilancia reranking e confronta i chunk (e la risposta, se vuoi) con la baseline
handler.run_top_j_distance_tag_reranking()
cmp = handler.compare_retrieval_vs_reranking()
print("baseline :", cmp["vectorial_baseline"]["chunk_ids"][:bench.top_j])
print("reranked :", cmp["post_reranking"]["chunk_ids"])
display(bench.edit_status().query("n_dist_edits > 0 or n_graph_tags > 0"))

In [ ]:
### Print query ###
baseline = cmp["vectorial_baseline"]
reranked = cmp["post_reranking"]

print("=" * 80)
print(f"???> Query: '{baseline['query']}'")
print("=" * 80)

print("\n --- 1. RISPOSTA BASELINE (1° Retrieval Vettoriale / Raw Records) ---")
print(f"  >> Chunk inclusi ({len(baseline['chunk_ids'])}): {baseline['chunk_ids']}")
print("\n  >> PROMPT INVIATO:")
print("-" * 40)
print(baseline["prompt"])
print("-" * 40)
print("\n  << RISPOSTA LLM:")
print(baseline["response"])

print("\n" + "=" * 80)

print("\n --- 2. RISPOSTA RERANKED (Post DistanceTagReranking / Top-J) ---")
print(f"  >> Chunk inclusi ({len(reranked['chunk_ids'])}): {reranked['chunk_ids']}")
print("\n  >> PROMPT INVIATO:")
print("-" * 40)
print(reranked["prompt"])
print("-" * 40)
print("\n  << RISPOSTA LLM:")
print(reranked["response"])
print("=" * 80)

In [ ]:
print(handler.query_tags)
for r in handler.rerank_generic_rslt["final_ranked"]:
    print(r.get("chunk_id"), r.get("initial_score"), r.get("tag_alteration"), r.get("predq_alteration"), r.get("final_score"), r.get("chunk_tags"))
display(bench.edit_status().query("n_dist_edits > 0"))